# Task 1: Financial AI Equity Research Pipeline Demonstration

This notebook serves as the execution and demonstration layer for the equity research pipeline.
All core logic resides inside the modular `src/` package.

## 1. Setup & Environment Initialization

In [ ]:
# %load_ext autoreload
# %autoreload 2

from src.config import config
print(f"Target Ticker: {config.DEFAULT_TICKER}")
print(f"Lookback Period: {config.LOOKBACK_YEARS} years")
print(f"LLM Model: {config.LLM_MODEL_NAME} (Temp: {config.LLM_TEMPERATURE})")

## 2. Market Data Retrieval & Technical Indicator Calculation (Phase 1 & Phase 2)

In [ ]:
from src.data.market_data import fetch_market_data
from src.features.technical_indicators import calculate_indicators

# Fetch 2 years of daily OHLCV data for target ticker
ticker_symbol = config.DEFAULT_TICKER
df_market = fetch_market_data(ticker_symbol, years=config.LOOKBACK_YEARS)
df_indicators = calculate_indicators(df_market)

print(f"Successfully calculated indicators for {ticker_symbol}. Shape: {df_indicators.shape}")
display(df_indicators.tail(5))

## 3. Financial Summary & News Ingestion (Phase 3)

In [ ]:
from src.data.news_data import fetch_news_data
from src.features.summary import generate_financial_summary

summary = generate_financial_summary(ticker_symbol, df_indicators)
news = fetch_news_data(ticker_symbol, limit=config.NEWS_HEADLINES_LIMIT)

print("Financial Summary:", summary)
print(f"Retrieved {len(news)} news headlines.")

## 4. LLM News Sentiment Analysis (Phase 4)

In [ ]:
from src.llm.sentiment import analyze_batch_sentiment
import pandas as pd

sentiment_summary = analyze_batch_sentiment(news, ticker_symbol)
print(f"Overall Sentiment Label: {sentiment_summary.overall_label}")
print(f"Weighted Sentiment Score: {sentiment_summary.weighted_sentiment_score}")
display(pd.DataFrame([r.model_dump() for r in sentiment_summary.headline_results]))

## 5. LLM Trading Recommendation & Cross-Indicator Reasoning (Phase 5)

In [ ]:
from src.llm.signal import generate_trading_signal

# Synthesize recommendation using pre-computed Python indicators
latest_row = df_indicators.iloc[-1]
recommendation = generate_trading_signal(
    ticker=ticker_symbol,
    summary=summary,
    latest_indicators=latest_row,
    news_sentiment=sentiment_summary
)

print(f"Recommendation: {recommendation.recommendation}")
print(f"Evidence-Based Reasoning:\n{recommendation.reasoning}")